# S23 · Press play on a bank queue with SimPy

Now we press play on a little model of an SBI branch with one teller. Customers arrive
at random moments, wait in line if the teller is busy, get served for a random time,
then leave. We run the clock and measure the average waiting time and the average
number of people in the queue. This is **discrete-event simulation**: a flight
simulator for a queue.

**New here? Read this once.**

- New to Python? You can still run every cell with the play button and read along. The
  new word here is `yield`, which just means "wait for this to happen before going on".
- New to the idea? Open `primers/the_law_of_large_numbers.md`; the last step here is
  the same "average many runs" idea, one level up.
- Already confident? Run the single-teller version, then look for the cells marked
  **Stretch (optional)** near the end (a hospital OPD with priority, and a second
  teller that makes the wait collapse).
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once to install SimPy (Colab does not ship it).
On your **own machine** you already installed everything with `uv`, so it does nothing
there.

In [ ]:
# SimPy is the one library here that Colab does not already include.
import sys
if "google.colab" in sys.modules:
    !pip install -q simpy
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — what SimPy does

SimPy is a Python library for **discrete-event simulation**. It keeps a clock and
jumps from one event to the next: an arrival, a service finishing. Nothing happens in
between, so it can simulate a whole working day in a blink. We also need NumPy for the
random arrival and service times, and a seed so the run is reproducible.

In [ ]:
import simpy          # the discrete-event simulation library
import numpy as np      # for the random arrival and service times

# Set a seed so the random times are the same every time we run this.
np.random.seed(0)

print("SimPy is ready.")

## Step 2 — choose the numbers that describe our bank

Two rates describe the branch:

- customers arrive on average every 5 minutes,
- the teller takes on average 4 minutes to serve one customer.

Because the teller is a little faster than the arrivals, the queue should stay under
control. We run the simulation for several working days (2400 minutes) so we serve
enough customers for the average to settle down.

In [ ]:
# Average gap between two arrivals, in minutes.
average_time_between_arrivals = 5.0

# Average time to serve one customer, in minutes.
average_service_time = 4.0

# How long to run the simulation, in minutes. Several working days (2400 minutes)
# so we serve enough customers for the average to be steady.
simulation_minutes = 2400

# We will collect every customer's waiting time in this list.
waiting_times = []

print("Set up: arrivals every", average_time_between_arrivals, "min on average,",
      "service", average_service_time, "min on average.")

## Step 3 — describe one customer's life

In SimPy each customer is a small Python function. It describes, step by step, what
one customer does:

1. note the time they arrived,
2. ask for the teller (and wait in line if the teller is busy),
3. once it is their turn, record how long they waited,
4. be served for a random amount of time,
5. leave, which frees the teller for the next person.

The word `yield` means *wait for this to happen* before going on.

In [ ]:
def customer(env, name, teller):
    # The simulation clock right now is when this customer arrived.
    arrival_time = env.now

    # Ask for the teller. "with" makes sure the teller is released at the end.
    with teller.request() as request_the_teller:
        # Wait here until the teller is free (this is the time spent in the queue).
        yield request_the_teller

        # The teller is now free; work out how long we waited.
        wait = env.now - arrival_time
        waiting_times.append(wait)

        # Being served takes a random time, drawn from an exponential distribution.
        service_time = np.random.exponential(average_service_time)
        yield env.timeout(service_time)
    # Leaving the "with" block frees the teller for the next customer.

## Step 4 — describe the stream of arrivals

A second small function keeps creating new customers. After each one it waits a random
gap (drawn from an exponential distribution, the waiting-time shape from notebook 2)
before the next customer arrives. That is what makes arrivals random and bunchy, just
like a real branch.

In [ ]:
def generate_arrivals(env, teller):
    customer_number = 0
    # Keep making customers for as long as the simulation runs.
    while True:
        # Wait a random gap before the next customer walks in.
        gap = np.random.exponential(average_time_between_arrivals)
        yield env.timeout(gap)

        # A new customer has arrived; give them a number and start their process.
        customer_number = customer_number + 1
        env.process(customer(env, customer_number, teller))

## Step 5 — run the simulation

We create the environment (the clock), create the teller as a resource with
**capacity 1** (one teller), start the arrivals, and run the clock for the whole
period.

In [ ]:
# The environment holds the simulation clock.
env = simpy.Environment()

# The teller is a resource with capacity 1: only one customer served at a time.
teller = simpy.Resource(env, capacity=1)

# Start the arrivals process.
env.process(generate_arrivals(env, teller))

# Run the clock for the whole simulation period (2400 minutes).
env.run(until=simulation_minutes)

print("Simulation finished.")
print("Number of customers served:", len(waiting_times))

## Step 6 — the average waiting time

Now we read off the result we care about: how long, on average, did a customer wait in
line before being served?

In [ ]:
# Turn the list of waits into a NumPy array so we can average it easily.
waiting_times_array = np.array(waiting_times)

average_wait = np.mean(waiting_times_array)
longest_wait = np.max(waiting_times_array)

print("average waiting time:", round(average_wait, 2), "minutes")
print("longest waiting time:", round(longest_wait, 2), "minutes")

## Step 7 — the average queue length

There is a neat shortcut called **Little's Law**: the average number of people in the
system equals the arrival rate times the average time each person spends in it. We use
it to turn our waiting times into an average queue length.

The arrival rate is `1 / average_time_between_arrivals` customers per minute. The time
in the system is the wait plus the service time.

In [ ]:
# Arrivals per minute (the rate at which customers come in).
arrival_rate = 1 / average_time_between_arrivals

# Average time a customer spends in the system = waiting + being served.
average_time_in_system = average_wait + average_service_time

# Little's Law: average number in the system = rate * time in system.
average_number_in_system = arrival_rate * average_time_in_system

print("average number of people in the system:", round(average_number_in_system, 2))

## Step 8 — picture the waiting times

The output of a simulation is not one number, it is a whole distribution. Most
customers wait a little, but a few wait a lot. The histogram shows this, and the
dashed line marks the average.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(waiting_times_array, bins=20, color="#2E75B6", edgecolor="white")
plt.axvline(average_wait, color="#C0392B", linestyle="--", linewidth=2,
            label="average wait")
plt.xlabel("waiting time (minutes)")
plt.ylabel("number of customers")
plt.title("Most customers wait a little; a few wait a lot")
plt.legend()
plt.show()

## Step 9 — one run is just one story

A single simulation run is like a single anecdote. Run it again with a different seed
and the average wait can change a lot. For a busy queue like this one, one run might
say 27 minutes and another 12 minutes, because randomness has a big effect when the
teller is almost always busy. So do not trust the single number from Step 6.

To get a trustworthy answer we **repeat** the whole simulation many times and average
the results, then report a **confidence interval**, an honest range for the true
value. This is the law of large numbers again, one level up: averaging over runs
instead of over darts.

In [ ]:
# Run the whole simulation 30 times, each with a different seed, and collect the
# average wait from each run.
average_wait_each_run = []

for run_number in range(30):
    # A different seed for each run gives a different set of random times.
    np.random.seed(run_number)

    # Reset the list of waits for this run.
    waiting_times = []

    # Build and run a fresh simulation, exactly as before.
    env = simpy.Environment()
    teller = simpy.Resource(env, capacity=1)
    env.process(generate_arrivals(env, teller))
    env.run(until=simulation_minutes)

    # Record this run's average wait.
    average_wait_each_run.append(np.mean(waiting_times))

average_wait_each_run = np.array(average_wait_each_run)

# The mean of the run-averages is our best estimate.
overall_average = np.mean(average_wait_each_run)

# A rough 95% confidence interval: mean plus/minus about 2 standard errors.
standard_error = np.std(average_wait_each_run) / np.sqrt(len(average_wait_each_run))
low = overall_average - 2 * standard_error
high = overall_average + 2 * standard_error

print("average wait over 30 runs:", round(overall_average, 2), "minutes")
print("95% confidence interval  :", round(low, 2), "to", round(high, 2), "minutes")

### Stretch (optional) — open a second counter (a Flipkart packing station)

Skip this if you are new to code. If you are comfortable, here is the manager's most
common fix: add a second server. Picture a Flipkart packing station where two packers
share one incoming stream of parcels. In SimPy that is a one-word change: give the
resource `capacity=2`. Everything else stays the same. Watch the average wait
collapse, because a customer now waits only until *either* server is free.

In [ ]:
# Run the same 30 replications, but with two servers sharing the queue.
two_server_wait_each_run = []

for run_number in range(30):
    np.random.seed(run_number)
    waiting_times = []            # reset the shared list for this run

    env = simpy.Environment()
    # capacity=2: two packers serving one queue.
    counter = simpy.Resource(env, capacity=2)
    env.process(generate_arrivals(env, counter))
    env.run(until=simulation_minutes)

    two_server_wait_each_run.append(np.mean(waiting_times))

two_server_wait_each_run = np.array(two_server_wait_each_run)

print("average wait, ONE server :", round(overall_average, 2), "minutes")
print("average wait, TWO servers:", round(np.mean(two_server_wait_each_run), 2), "minutes")
print()
print("One extra server does not just halve the wait - it can nearly erase it,")
print("because the queue no longer stalls whenever the single server is busy.")

### Stretch (optional) — a hospital OPD, where urgent patients jump the queue

No code to run here, just the idea for you to try on your own. A bank serves customers
first-come-first-served. A hospital OPD does not: a patient in critical condition is
seen before someone with a mild complaint, however long the mild case has waited. That
is a **priority queue**.

SimPy models it with `simpy.PriorityResource` instead of `simpy.Resource`, and each
`request(priority=...)` carries how urgent that patient is (smaller numbers go first).
The interesting question becomes: with urgent patients jumping ahead, do the
non-urgent ones wait dangerously long? Rebuild this notebook with a
`PriorityResource`, give each arrival a random priority, and measure the wait
separately for urgent and routine patients. That is a genuinely useful model of a
government hospital OPD.

## What you just did

You built a working discrete-event simulation of a bank queue with SimPy: customers
arrive at random, wait for a single teller, are served for a random time, and leave.
You measured the average wait and queue length, and, most importantly, you learned to
repeat the run many times and report a confidence interval, because a single run is
only one story.

Try changing the service time to 4.8 minutes and watch the queue explode: when the
teller is busy almost all the time, waits blow up. That cliff, where **utilisation**
nears 100%, is the single most important thing simulation reveals about a queue.